# 9. 시군구 단위 리스테리아 검출률 — 전체 vs 버섯 제외

- 시군구 = `legal_dong_code` 앞 5자리 (시·도 2 + 시·군·구 3)
- `legal_dong_code` 결측 행은 시군구 집계에서 제외
- 버섯 행 = `prduct_nm` 에 버섯·MUSHROOM·Flammulina·ENOKI·팽이 포함 (대소문자 무시)
- 시군구 이름 = `addr` 앞 두 단어의 최빈값

In [1]:
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 400)

df = pd.read_parquet(r"C:\cheon\cheon_wokespace\listeria2026\리스테리아ML\data\listeria_body.parquet")
_code = df["legal_dong_code"].astype(object)
df["sgg"] = _code.where(_code.notna()).astype(str).str[:5].where(_code.notna())
df["pos"] = (df["judge_result"] == "검출").astype(int)
df["is_mush"] = df["prduct_nm"].astype(str).str.contains("버섯|MUSHROOM|Flammulina|ENOKI|팽이", case=False, regex=True)

print(f"전체 {len(df):,}행 / 검출 {df['pos'].sum():,}건 / 검출률 {df['pos'].mean() * 100:.3f}%")
print(f"legal_dong_code 결측 {df['sgg'].isna().sum():,}행 (검출 {df.loc[df['sgg'].isna(), 'pos'].sum()}건) → 시군구 집계 제외")
print(f"버섯 행 {df['is_mush'].sum():,}행 / 검출 {df.loc[df['is_mush'], 'pos'].sum()}건")
print(f"버섯 제외 검출률(시군구 있는 행): {df.loc[~df['is_mush'] & df['sgg'].notna(), 'pos'].mean() * 100:.3f}%")

전체 676,537행 / 검출 1,969건 / 검출률 0.291%
legal_dong_code 결측 47,642행 (검출 125건) → 시군구 집계 제외
버섯 행 9,851행 / 검출 526건
버섯 제외 검출률(시군구 있는 행): 0.213%


## 9-1. 시군구별 행수·검출·검출률 — 전체 / 버섯 제외 나란히

In [2]:
d = df[df["sgg"].notna()]
sgg_nm = (d.assign(_nm=d["addr"].astype(str).str.split().str[:2].str.join(" "))
           .groupby("sgg")["_nm"].agg(lambda s: s.mode().iat[0]))


def sgg_tbl(x):
    t = x.groupby("sgg").agg(행수=("pos", "size"), 검출=("pos", "sum"))
    t["검출률(%)"] = (t["검출"] / t["행수"] * 100).round(3)
    t["검출비중(%)"] = (t["검출"] / t["검출"].sum() * 100).round(2)   # 그 그룹 총 검출(전체 1,844 / 버섯제외 1,318) 중 이 시군구 몫
    return t


sgg = pd.concat({"전체": sgg_tbl(d), "버섯제외": sgg_tbl(d[~d["is_mush"]])}, axis=1)
sgg.insert(0, "시군구명", sgg_nm)
sgg = sgg.sort_values(("전체", "검출"), ascending=False)
print(f"시군구 {len(sgg)}개 / 검출 1건 이상 {(sgg[('전체', '검출')] > 0).sum()}개")
print(f"검출비중 분모: 전체 {int(sgg[('전체', '검출')].sum()):,}건 / 버섯제외 {int(sgg[('버섯제외', '검출')].sum()):,}건 (legal_dong_code 있는 행)")
display(sgg)

시군구 288개 / 검출 1건 이상 158개
검출비중 분모: 전체 1,844건 / 버섯제외 1,318건 (legal_dong_code 있는 행)


시군구명     전체                        버섯제외                    
                        행수   검출  검출률(%) 검출비중(%)     행수   검출 검출률(%) 검출비중(%)
sgg                                                                       
47820      경상북도 청도군   7295  327   4.483   17.73   4237   23  0.543    1.75
41610       경기도 광주시  21969  109   0.496    5.91  21899  109  0.498    8.27
43770      충청북도 음성군  20968  100   0.477    5.42  20172   28  0.139    2.12
11650     서울특별시 서초구   3259   86   2.639    4.66   2526   25  0.990    1.90
41590       경기도 화성시  12144   64   0.527    3.47  12128   64  0.528    4.86
46720      전라남도 곡성군    962   42   4.366    2.28    841   22  2.616    1.67
41133       경기도 성남시   8905   38   0.427    2.06   8851   38  0.429    2.88
27230      대구광역시 북구   2619   36   1.375    1.95   2618   36  1.375    2.73
41550       경기도 안성시  13525   36   0.266    1.95  13508   36  0.267    2.73
29200     광주광역시 광산구   3764   35   0.930    1.90   3759   35  0.931    2.66
41450       경기도 하남시   5327   33   0.619    1.79   5325   33  0.620    2.50
41461       경기도 용인시  12509   32   0.256    1.74  12490   32  0.256    2.43
41480       경기도 파주시  14620   31   0.212    1.68  14612   31  0.212    2.35
11200     서울특별시 성동구   8092   29   0.358    1.57   8058   29  0.360    2.20
28260      인천광역시 서구   7624   28   0.367    1.52   7599   28  0.368    2.12
46170      전라남도 나주시   5766   25   0.434    1.36   5361   10  0.187    0.76
41500       경기도 이천시  10129   24   0.237    1.30  10126   24  0.237    1.82
41570       경기도 김포시   7924   24   0.303    1.30   7903   23  0.291    1.75
41650       경기도 포천시  15487   24   0.155    1.30  15445   24  0.155    1.82
51210       강원도 속초시   3718   23   0.619    1.25   3642   23  0.632    1.75
50110   제주특별자치도 제주시   8573   22   0.257    1.19   8568   21  0.245    1.59
28200     인천광역시 남동구   4724   19   0.402    1.03   4696   19  0.405    1.44
47290      경상북도 경산시   4268   18   0.422    0.98   4265   16  0.375    1.21
41390       경기도 시흥시   3574   18   0.504    0.98   3574   18  0.504    1.37
41360      경기도 남양주시  12423   17   0.137    0.92  12400   17  0.137    1.29
47230      경상북도 영천시   2312   17   0.735    0.92   2307   17  0.737    1.29
41463       경기도 용인시   2964   16   0.540    0.87   2956   16  0.541    1.21
43114      충청북도 청주시   2993   15   0.501    0.81   2971   15  0.505    1.14
52180      전라북도 정읍시   3937   15   0.381    0.81   3937   15  0.381    1.14
11545     서울특별시 금천구   1547   13   0.840    0.70   1542   13  0.843    0.99
41196       경기도 부천시    777   13   1.673    0.70    776   13  1.675    0.99
52140   전북특별자치도 익산시   7426   13   0.175    0.70   7414   13  0.175    0.99
43130      충청북도 충주시   4124   12   0.291    0.65   4124   12  0.291    0.91
44230      충청남도 논산시   8337   12   0.144    0.65   8333   12  0.144    0.91
50130  제주특별자치도 서귀포시   2418   12   0.496    0.65   2405   12  0.499    0.91
28245     인천광역시 계양구    872   12   1.376    0.65    871   12  1.378    0.91
47000      경상북도 이서면   1367   12   0.878    0.65    240    5  2.083    0.38
44133      충청남도 천안시   8620   11   0.128    0.60   8591   11  0.128    0.83
41630       경기도 양주시  11255   11   0.098    0.60  11111   11  0.099    0.83
43113      충청북도 청주시   4113   11   0.267    0.60   4024    2  0.050    0.15
11710     서울특별시 송파구   1789   11   0.615    0.60   1782   11  0.617    0.83
48720      경상남도 의령군   1048   10   0.954    0.54    404    0  0.000    0.00
47280      경상북도 문경시   2186   10   0.457    0.54   2186   10  0.457    0.76
47190      경상북도 구미시   3557   10   0.281    0.54   2695    6  0.223    0.46
29170      광주광역시 북구   4023   10   0.249    0.54   4023   10  0.249    0.76
27140      대구광역시 동구   1944    9   0.463    0.49   1940    9  0.464    0.68
47830      경상북도 고령군    552    9   1.630    0.49    552    9  1.630    0.68
48870      경상남도 함양군    526    8   1.521    0.43    514    1  0.195    0.08
43150      충청북도 제천시   2862    8   0.280    0.43   2860    8  0.280    0.61
46860      전라남도 함평군   1179    8   0.679    0.43    991    0  0.000    0.00
41271       경기도 안산시   1419    8   0.564 

## 9-1-1. 청도군(47820) 검출 327건 — 어떤 품목·제품·업체에서 나왔나

- 검출 행만 대상. 품목 = `train_prdlst_nm`, 제품 = `prduct_nm`, 업체 = `mnfctur_cmpny_nm`

In [3]:
cd = d[(d["sgg"] == "47820") & (d["pos"] == 1)].copy()
print(f"청도군 검출 {len(cd):,}건 / 버섯 행 {int(cd['is_mush'].sum())}건 / 버섯 아님 {int((~cd['is_mush']).sum())}건")

print("[1] 품목(train_prdlst_nm) × 버섯 여부")
display(pd.crosstab(cd["train_prdlst_nm"].astype(str), cd["is_mush"].map({True: "버섯", False: "버섯 아님"}), margins=True, margins_name="합계"))

print("[1-1] 같은 표를 청도군 검출 327건 대비 비율(%)로")
display((pd.crosstab(cd["train_prdlst_nm"].astype(str), cd["is_mush"].map({True: "버섯", False: "버섯 아님"}),
                     margins=True, margins_name="합계", normalize="all") * 100).round(1))

print("[2] 업체(mnfctur_cmpny_nm)별 검출 건수 / 주소 출처(addr_source)")
display(cd["mnfctur_cmpny_nm"].astype(str).value_counts().rename("검출건수").to_frame())
display(cd["addr_source"].astype(str).value_counts().rename("검출건수").to_frame())

print("[3] 연도별 검출 건수 (mnfctur_de 기준)")
display(cd["mnfctur_de"].dt.year.value_counts().sort_index().rename("검출건수").to_frame())

print("[4] 제품명(prduct_nm)별 검출 건수 — 전부")
display(cd.groupby([cd["is_mush"].map({True: "버섯", False: "버섯 아님"}), cd["prduct_nm"].astype(str)])
          .size().rename("검출건수").sort_values(ascending=False).to_frame())

print("[5] 버섯 키워드에 안 걸린 검출 행 상세")
display(cd.loc[~cd["is_mush"], ["train_prdlst_nm", "prduct_nm", "mnfctur_cmpny_nm", "mnfctur_de", "addr"]].sort_values("mnfctur_de"))

청도군 검출 327건 / 버섯 행 304건 / 버섯 아님 23건
[1] 품목(train_prdlst_nm) × 버섯 여부


is_mush,버섯,버섯 아님,합계
train_prdlst_nm,,,
기타기준규격외,2,14,16
"정액,수정란,종란",302,9,311
합계,304,23,327


[1-1] 같은 표를 청도군 검출 327건 대비 비율(%)로


is_mush,버섯,버섯 아님,합계
train_prdlst_nm,,,
기타기준규격외,0.6,4.3,4.9
"정액,수정란,종란",92.4,2.8,95.1
합계,93.0,7.0,100.0


[2] 업체(mnfctur_cmpny_nm)별 검출 건수 / 주소 출처(addr_source)


,검출건수
mnfctur_cmpny_nm,
농업회사법인(주)머쉬앤파머스,59
농업회사법인 주식회사 머쉬앤파머스,14
농업회사법인 구라 주식회사,4
명덕농산,4
농업회사법인 학암 주식회사,3
농업회사법인 닥터애그주식회사,3


... (3 outputs left out)

[4] 제품명(prduct_nm)별 검출 건수 — 전부


검출건수
is_mush prduct_nm                                               
버섯      ENOKI MUSHROOM GR4-2                                   6
        수출용 팽이버섯 (2번)                                          6
        수출용 팽이버섯 (3번)                                          6
        ENOKI MUSHROOM GR5-3                                   6
        ENOKI MUSHROOM GR4-1                                   6
        ENOKI MUSHROOM GR1-3                                   5
        수출용 팽이버섯 (2-2번)                                        5
        ENOKI MUSHROOM GR6-1                                   5
        ENOKI MUSHROOM GR4-3                                   5
        수출용 팽이버섯 (7번)                                          5
        수출용 팽이버섯 (1번)                                          4
        ENOKI MUSHROOM GR3-1                                   4
        ENOKI MUSHROOM GR2-1                                   4
        ENOKI MUSHROOM GR7-1                                   4
        ENOKI MUSHROOM GR3-2                                   4
        ENOKI MUSHROOM GR5-1                                   4
        ENOKI MUSHROOM GR5-5                                   3
        ENOKI MUSHROOM GR03                                    3
        ENOKI MUSHROOM GR6-3                                   3
        수출용 팽이버섯 (2-3번)                                        3
        ENOKI MUSHROOM 수확후-5                                   3
        수출용 팽이버섯 (9번)                                          3
        ENOKI MUSHROOM 수확후-10                                  3
        ENOKI MUSHROOM GR8-3                                   3
        ENOKI MUSHROOM GR8-5                                   3
        ENOKI MUSHROOM GR01                                    3
        수출용 팽이버섯 (6번)                                          3
        ENOKI MUSHROOM GR5-4                                   3
        ENOKI MUSHROOM GR1-1                                   3
        ENOKI MUSHROOM GR3-3                                   3
        ENOKI MUSHROOM GR06                                    3
        ENOKI MUSHROOM GR1-4                                   3
        Flammulina velutipes(Enoki mushroom) ONE.S 150g -5     3
        ENOKI MUSHROOM GR1-5                                   3
        ENOKI MUSHROOM GR2-2                                   3
        ENOKI MUSHROOM GR3-4                                   3
        ENOKI MUSHROOM GR2-3                                   3
        ENOKI MUSHROOM GR2-4                                   3
        ENOKI MUSHROOM 수확후-4                                   2
        수출용 팽이버섯 (8번)                                          2
        Flammulina velutipes(Enoki mushroom) ONE.S 150g...     2
        ENOKI MUSHROOM 수확후-1                                   2
버섯 아님   1-5                                                    2
버섯      Flammulina velutipes(Enoki mushroom) ONE.S 150g...     2
버섯 아님   1-15                                                   2
버섯      팽이버섯                                                   2
버섯 아님   1-10                                                   2
버섯      수출용 팽이버섯 (2-1번)                                        2
        ENOKI MUSHROOM 수확전-7                                   2
        ENOKI MUSHROOM GG03                                    2
        Flammulina velutipes(Enoki mushroom) ONE.S 150g...     2
        ENOKI MUSHROOM 수확후-9                                   2
        ENOKI MUSHROOM GR8-2                                   2
        ENOKI MUSHROOM 수확전-4                                   2
        ENOKI MUSHROOM GR5-2                                   2
        ENOKI MUSHROOM GR02                                    2
        ENOKI MUSHROOM GR05                                    2
        ENOKI MUSHROOM GR07                                    2
        ENOKI MUSHROOM GR08                                    2
        ENOKI MUSHROOM GR1-2                                   2
        ENOKI MUSHROOM GR3-5  

[5] 버섯 키워드에 안 걸린 검출 행 상세


,train_prdlst_nm,prduct_nm,mnfctur_cmpny_nm,mnfctur_de,addr
490368,기타기준규격외,1-5,<NA>,2021-03-08,경상북도 청도군 금천면 소천길 271 1층
271595,기타기준규격외,2-4,<NA>,2021-03-29,경상북도 청도군 금천면 소천길 271 1층
520431,기타기준규격외,1-15,<NA>,2021-04-19,경상북도 청도군 금천면 소천길 271 1층
530318,기타기준규격외,참깨 - 가열 후,<NA>,2021-07-26,경상북도 청도군 금천면 죽전길 7-1
351669,기타기준규격외,1-14,<NA>,2021-09-13,경상북도 청도군 금천면 소천길 271 1층
609668,기타기준규격외,1-5,<NA>,2021-09-13,경상북도 청도군 금천면 소천길 271 1층
252319,기타기준규격외,1-13,<NA>,2021-09-27,경상북도 청도군 금천면 소천길 271 1층
13614,기타기준규격외,원스150g-31,<NA>,2021-10-12,경상북도 청도군 금천면 소천길 271 1층
501147,기타기준규격외,원스150g-12,<NA>,2021-10-12,경상북도 청도군 금천면 소천길 271 1층
629435,기타기준규격외,E&B 1-21,<NA>,2021-10-18,경상북도 청도군 금천면 소천길 271 1층


In [4]:
print("[6] 청도군 버섯 행 — 주소 출처(addr_source) value_counts")
cm = d[(d["sgg"] == "47820") & d["is_mush"]]
src = pd.concat({"전체 버섯 행": cm["addr_source"].astype(str).value_counts(),
                 "검출 버섯 행": cm.loc[cm["pos"] == 1, "addr_source"].astype(str).value_counts()}, axis=1).fillna(0).astype(int)
src.loc["합계"] = src.sum()
display(src)

[6] 청도군 버섯 행 — 주소 출처(addr_source) value_counts


,전체 버섯 행,검출 버섯 행
addr_source,,
제조사소재지,1773,78
업소주소,1285,226
합계,3058,304
